# 第8回講義 宿題



## 課題
ViTによる画像分類を実装してみましょう．


### 目標値
なし
- 今回は計算リソースによってモデルの性能が大きく変わるため，目標精度は設定していません．

### ルール
- 訓練データは`x_train`， `t_train`，テストデータは`x_test`で与えられます．
- 予測ラベルは one_hot表現ではなく0~9のクラスラベル で表してください．
- **下のセルで指定されている`x_train`，`t_train`以外の学習データは使わないでください．**
- **演習用に配布されている`trained_vision_model.pth`は使わないでください．**


### 提出方法
- 2つのファイルを提出していただきます．
    1. テストデータ (`x_test`) に対する予測ラベルを`submission_pred.csv`として保存し，**Omnicampusの宿題タブから「第8回 Transformer基礎」を選択して**提出してください．
    2. それに対応するpythonのコードを`submission_code.py`として保存し，**Omnicampusの宿題タブから「第8回 Transformer基礎 (code)」を選択して**提出してください．pythonファイル自体の提出ではなく，「提出内容」の部分にコードをコピー&ペーストしてください．
      
- なお，採点は1で行い，2はコードの確認用として利用します（成績優秀者はコード内容を公開させていただくかもしれません）．コードの内容を変更した場合は，**1と2の両方を提出し直してください**．


### 評価方法
- 予測ラベルの`t_test`に対する精度 (Accuracy) で評価します．
- 即時採点しLeader Boardを更新します．
- 締切時の点数を最終的な評価とします．

### ドライブのマウント

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 作業ディレクトリを指定
work_dir = '/content/drive/MyDrive/DLBasic/HW/HW8'

### データの読み込み（このセルは修正しないでください）

In [ ]:
!sudo apt update
!sudo apt install xvfb

import sys
import random
import math
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from PIL import Image
import h5py
from os.path import join
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import Dataset
from torch.utils.data.dataloader import DataLoader
from torchvision import datasets, transforms
from einops.layers.torch import Rearrange
from einops import rearrange, repeat
import timm

import logging

import torch.optim as optim
from torch.optim.lr_scheduler import LambdaLR
from torch.nn import functional as F

seed=42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.set_printoptions(edgeitems=1e3)

# 要素にドットでアクセスできる辞書クラス
class Args(dict):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.__dict__ = self

#学習データ
x_train = np.load(work_dir + '/Lecture08/data/x_train.npy')
t_train = np.load(work_dir + '/Lecture08/data/t_train.npy')

#テストデータ
x_test = np.load(work_dir + '/Lecture08/data/x_test.npy')

class train_dataset(torch.utils.data.Dataset):
    def __init__(self, x_train, t_train):
        data = x_train.astype('float32')
        self.x_train = []
        for i in range(data.shape[0]):
            self.x_train.append(Image.fromarray(np.uint8(data[i])))
        self.t_train = t_train
        self.transform = transforms.ToTensor()

    def __len__(self):
        return len(self.x_train)

    def __getitem__(self, idx):
        return self.transform(self.x_train[idx]), torch.tensor(t_train[idx], dtype=torch.long)

class test_dataset(torch.utils.data.Dataset):
    def __init__(self, x_test):
        data = x_test.astype('float32')
        self.x_test = []
        for i in range(data.shape[0]):
            self.x_test.append(Image.fromarray(np.uint8(data[i])))
        self.transform = transforms.ToTensor()

    def __len__(self):
        return len(self.x_test)

    def __getitem__(self, idx):
        return self.transform(self.x_test[idx])

trainval_data = train_dataset(x_train, t_train)
test_data = test_dataset(x_test)

Hit:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [3,296 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,557 kB]
Fetched 5,238 kB in 4s (1,411 kB/s)
Reading package lists... Done
Building dependency tr

### データセットの準備  

In [ ]:
val_size = 3000
train_data, valid_data = torch.utils.data.random_split(trainval_data, [len(trainval_data) - val_size, val_size])

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))]
)

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))]
)

trainval_data.transform = train_transform
test_data.transform = test_transform

### ViTの実装

In [ ]:
class ViT(nn.Module):
    def __init__(self, config, pretrained=True, freeze_backbone=False) -> None:
        """
        Parameters
        ----------
        pretrained(bool): 事前訓練済みViTモデルを使用するか
        freeze_backbone(bool): バックボーンを凍結するか
        """
        super().__init__()

        model_name = getattr(config, 'model_name', 'vit_small_patch16_224')

        self.backbone = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=0, # ヘッドの部分を無効化
            global_pool='token' # clsトークンを使って特徴を集約
        )

        # バックボーンの特徴量の次元を取得
        with torch.no_grad():
            dummy_input = torch.randn(1, 3, 224, 224)
            features = self.backbone(dummy_input)
            feature_dim = features.shape[-1]

        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False

        self.classifier = nn.Sequential(
              nn.LayerNorm(feature_dim),
              nn.Dropout(0.1),
              nn.Linear(feature_dim, feature_dim // 2),
              nn.GELU(),
              nn.Dropout(0.1),
              nn.Linear(feature_dim // 2, config.n_class),
        )

        # backboneが期待する画像の形Wを取得
        self.input_size = self.backbone.default_cfg['input_size'][-1] # vit_small_patch16_224: 224

    def forward(self, x, target):
        # 入力画像のサイズを事前学習モデルのサイズにリサイズ
        if x.shape[-1] != self.input_size:
            x = F.interpolate(x, size=(self.input_size, self.input_size), mode='bilinear', align_corners=False)

        features = self.backbone(x)

        logits = self.classifier(features)

        if target is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), target.view(-1))
        else:
            loss = None

        return logits, loss

    def configure_optimizers(self, train_config):
        """
        バックボーンとヘッドで異なる学習率を設定
        """
        backbone_params = []
        head_params = []

        for name, param in self.named_parameters():
            if param.requires_grad:
                if 'backbone' in name:
                    backbone_params.append(param)
                else:
                    head_params.append(param)

        param_groups = [
              {'params': backbone_params, 'lr': train_config.learning_rate * 0.1},
              {'params': head_params, 'lr': train_config.learning_rate}
        ]

        optimizer = torch.optim.AdamW(
              param_groups,
              betas=train_config.betas,
              weight_decay=train_config.weight_decay,
        )

        return optimizer

class ProgressiveUnfreezing:
    """
    段階的にレイヤーを解凍する
    """
    def __init__(self, model, unfreeze_schedule):
        self.model = model
        self.unfreeze_schedule = unfreeze_schedule

    def step(self, epoch):
        if epoch in self.unfreeze_schedule:
            num_layers = self.unfreeze_schedule[epoch]
            self._unfreeze_layers(num_layers)

    def _unfreeze_layers(self, num_layers):
        blocks = list(self.model.backbone.blocks)
        # 最後のnum_layers個のブロックを解凍
        for i in range(max(0, len(blocks) - num_layers), len(blocks)):
            for param in blocks[i].parameters():
                param.requires_grad = True
        print(f"Unfreeze last {num_layers} blocks")

class MixupAugmentation:
    """
    mixup data augmentation
    """
    def __init__(self, alpha=0.2):
        self.alpha = alpha

    def __call__(self, batch_x, batch_y):
        if self.alpha > 0:
            # beta分布からlamをサンプリング(混ぜ具合)
            lam = torch.distributions.Beta(self.alpha, self.alpha).sample()
            batch_size = batch_x.size(0)
            index = torch.randperm(batch_size)

            mixed_x = lam * batch_x + (1 - lam) * batch_x[index]
            y_a, y_b = batch_y, batch_y[index]

            return mixed_x, y_a, y_b, lam
        return batch_x, batch_y, None, None


In [ ]:
# Trainerを設定

logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)

class TrainerConfig:
    # 最適化のパラメータ
    max_epochs = 10
    batch_size = 64
    learning_rate = 3e-4
    betas = (0.9, 0.95)
    grad_norm_clip = 1.0
    weight_decay = 0.1  # 行列乗算に使用する重みにのみ適用
    # 学習率の減衰パラメータ：線形warmupの後、元の学習率の10%までcosine減衰
    lr_decay = False
    warmup_tokens = 375e6  # warmup_tokensとfinal_tokensの値はGPT-3論文に由来するが，他のケースでも適切な初期値とは限らない
    final_tokens = 260e9  # このトークン数を処理した時点で，学習率が始めの値の10%まで下がるようにする
    # チェックポイントの設定
    ckpt_path = None
    num_workers = 0  # DataLoader用

    def __init__(self, **kwargs):
        for k,v in kwargs.items():
            setattr(self, k, v)

class Trainer:

    def __init__(self, model, train_dataset, test_dataset, config):
        self.model = model
        self.train_dataset = train_dataset
        self.test_dataset = test_dataset
        self.config = config

        # システム上にあるすべてのGPUを使用
        self.device = 'cpu'
        if torch.cuda.is_available():
            self.device = torch.cuda.current_device()
            self.model = torch.nn.DataParallel(self.model).to(self.device)

        self.mixup = MixupAugmentation(alpha=0.2)

        unfreeze_schedule = {
              3: 2,
              6: 4,
              9: 6,
        }

        raw_model = model.module if hasattr(model, "module") else model
        self.progressive_unfreezing = ProgressiveUnfreezing(raw_model, unfreeze_schedule)

    def mixup_criterion(self, pred, y_a, y_b, lam):
        return lam * F.cross_entropy(pred, y_a) + (1 - lam) * F.cross_entropy(pred, y_b)

    def save_checkpoint(self):
        # DataParallel Wrapperは生のモデルのオブジェクトを.moduleに保持する
        raw_model = self.model.module if hasattr(self.model, "module") else self.model
        logger.info("saving %s", self.config.ckpt_path)
        torch.save(raw_model.state_dict(), self.config.ckpt_path)

    def train(self):
        model, config = self.model, self.config
        raw_model = model.module if hasattr(self.model, "module") else model
        optimizer = raw_model.configure_optimizers(config)

        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=config.max_epochs//3, T_mult=1, eta_min=1e-6
        )

        def run_epoch(split, epoch):
            is_train = split == 'train'
            model.train(is_train)
            data = self.train_dataset if is_train else self.test_dataset
            shuffle = is_train
            loader = DataLoader(
                data,
                shuffle=shuffle,
                pin_memory=True,
                batch_size=config.batch_size,
                num_workers=config.num_workers,
                drop_last=is_train,
            )

            losses = []
            correct = 0
            total = 0

            pbar = tqdm(enumerate(loader), total=len(loader)) if is_train else enumerate(loader)

            for it, (x, y) in pbar:

                # データを適切なデバイスに配置
                x = x.to(self.device)
                y = y.to(self.device)

                # 順伝播
                with torch.set_grad_enabled(is_train):
                    if is_train and torch.rand(1).item() < 0.3: # 30%の確率でMixup
                        mixed_x, y_a, y_b, lam = self.mixup(x, y)
                        logits, _ = model(mixed_x, None)
                        loss = self.mixup_criterion(logits, y_a, y_b, lam)
                    else:
                        logits, loss = model(x, y)

                    if hasattr(model, 'module'):
                        loss = loss.mean()  # 複数GPUに分散している場合損失をまとめる
                    losses.append(loss.item())

                    # calculate accuracy
                    _, predicted = torch.max(logits.data, 1)
                    total += y.size(0)
                    correct += (predicted == y).sum().item()

                if is_train:

                    # 逆伝播およびパラメータ更新
                    model.zero_grad()
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_norm_clip)
                    optimizer.step()

            accuracy = 100. * correct / total
            avg_loss = sum(losses) / len(losses)

            if is_train:
                scheduler.step()
                # 進捗の表示
                logger.info(f"epoch {epoch+1} iter {it}: train loss {avg_loss:.5f}, train acc: {accuracy:.2f}%.")
            else:
                test_loss = float(np.mean(losses))
                logger.info(f"Epoch {epoch+1}: Val Loss: {avg_loss:.5f}, Val Acc: {accuracy:.2f}%")
                return avg_loss, accuracy

        # training loop
        best_acc = 0
        for epoch in range(config.max_epochs):

            self.progressive_unfreezing.step(epoch)

            run_epoch('train', epoch)
            if self.test_dataset is not None:
                val_loss, val_acc = run_epoch('test', epoch)

                if val_acc > best_acc:
                    best_acc = val_acc
                    if hasattr(self.config, 'ckpt_path') and self.config.ckpt_path:
                        self.save_checkpoint()


In [ ]:
block_size = 256

args = Args({
    'model_name': 'vit_small_patch16_224',
    'image_size': [32, 32],
    'patch_size': [2, 2],
    'n_layer': 4,
    'n_head': 8,
    'n_embd': 512,
    'n_class': 10,
})

model = ViT(args, pretrained=True, freeze_backbone=True)  # あとでtrainerがモデルをGPUに移してくれる

model_path = work_dir + '/Lecture08/models/trained_vision_model_homework.pth'

# Trainerをインスタンス化し, 訓練を開始
tconf = TrainerConfig(
    max_epochs=10,
    batch_size=32,
    learning_rate=1e-3,
    weight_decay=0.01,
    grad_norm_clip=1.0,
    num_workers=2,
    ckpt_path=model_path
)

trainer = Trainer(model, train_data, valid_data, tconf)


In [ ]:
# 学習
trainer.train()

# 学習したパラメータの保存
torch.save(model.state_dict(), model_path)

  0%|          | 0/1468 [00:00<?, ?it/s]

INFO:__main__:epoch 1 iter 1467: train loss 0.39918, train acc: 80.26%.
INFO:__main__:Epoch 1: Val Loss: 0.17740, Val Acc: 93.90%
INFO:__main__:saving /content/drive/MyDrive/DLBasic/HW/HW8/Lecture08/models/trained_vision_model_homework.pth


  0%|          | 0/1468 [00:00<?, ?it/s]

INFO:__main__:epoch 2 iter 1467: train loss 0.35162, train acc: 80.98%.
INFO:__main__:Epoch 2: Val Loss: 0.17587, Val Acc: 94.57%
INFO:__main__:saving /content/drive/MyDrive/DLBasic/HW/HW8/Lecture08/models/trained_vision_model_homework.pth


  0%|          | 0/1468 [00:00<?, ?it/s]

...

  0%|          | 0/1468 [00:00<?, ?it/s]

INFO:__main__:epoch 9 iter 1467: train loss 0.27191, train acc: 84.52%.
INFO:__main__:Epoch 9: Val Loss: 0.15142, Val Acc: 94.67%


Unfreeze last 6 blocks


  0%|          | 0/1468 [00:00<?, ?it/s]

INFO:__main__:epoch 10 iter 1467: train loss 0.28224, train acc: 82.96%.
INFO:__main__:Epoch 10: Val Loss: 0.15932, Val Acc: 94.73%


In [ ]:
# 評価の準備
device = "cuda" if torch.cuda.is_available() else "cpu"

# 学習したパラメータの読み込み
model.load_state_dict(torch.load(model_path))
model.eval();

In [ ]:
# # datasetをdata loaderにする
# train_dataloader = DataLoader(train_data, shuffle=True, pin_memory=True,
#                               batch_size=tconf.batch_size, num_workers=tconf.num_workers)
# valid_dataloader = DataLoader(valid_data, shuffle=False, pin_memory=True,
#                              batch_size=tconf.batch_size, num_workers=tconf.num_workers)

# train_acc, valid_acc = 0., 0.
# with torch.no_grad():
#     for x, y in train_dataloader:
#         x, y = x.to(device), y.to(device)
#         logits, _ = model(x, y)

#         acc = (torch.argmax(logits, dim=1) == y).float().sum().cpu()
#         train_acc += acc

#     for x, y in valid_dataloader:
#         x, y = x.to(device), y.to(device)
#         logits, _ = model(x, y)

#         acc = (torch.argmax(logits, dim=1) == y).float().sum().cpu()
#         valid_acc += acc

# print(f"Train Acc.: {(train_acc / len(train_data)):.4f}")
# print(f"Valid Acc. : {(valid_acc / len(valid_data)):.4f}")

In [ ]:
save = input("Do you want to save the result? (y/n): ")

Do you want to save the result? (y/n): y


In [ ]:
if save:
    test_dataloader = DataLoader(test_data, shuffle=False, pin_memory=True,
                             batch_size=tconf.batch_size, num_workers=tconf.num_workers)

    t_pred = []
    with torch.no_grad():
        for x in test_dataloader:
            x = x.to(device)
            logits, _ = model(x, None)

            # モデルの出力を予測値のスカラーに変換
            pred = logits.argmax(1).tolist()
            t_pred.extend(pred)

    submission = pd.Series(t_pred, name='label')
    submission.to_csv(work_dir + '/Lecture08/submission_pred.csv', header=True, index_label='id')